# Titanic V2 - 深度优化

多维度优化：特征工程 + 超参调优 + 模型融合

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, StackingClassifier)
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 合并处理，保持一致性
train['is_train'] = 1
test['is_train'] = 0
test['Survived'] = np.nan
full = pd.concat([train, test], sort=False).reset_index(drop=True)
print(f'Combined shape: {full.shape}')

## 1. 高级特征工程

In [ ]:
def advanced_feature_engineering(df):
    df = df.copy()
    
    # === 1. 称谓提取 ===
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    title_map = {
        'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
        'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', 'Don': 'Rare',
        'Dona': 'Rare', 'Lady': 'Rare', 'Countess': 'Rare',
        'Jonkheer': 'Rare', 'Sir': 'Rare', 'Capt': 'Rare'
    }
    df['Title'] = df['Title'].map(title_map).fillna('Rare')
    
    # === 2. 姓名长度 (社会地位代理) ===
    df['NameLen'] = df['Name'].apply(len)
    
    # === 3. 家庭特征 ===
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # 家庭规模分类
    df['FamilyCat'] = df['FamilySize'].map(lambda x: 'Single' if x == 1
                                          else 'Small' if x <= 4
                                          else 'Large')
    
    # === 4. 票号特征 ===
    # 提取票号前缀
    df['TicketPrefix'] = df['Ticket'].apply(lambda x: x.split()[0] if len(x.split()) > 1 else 'NONE')
    df['TicketPrefix'] = df['TicketPrefix'].str.replace(r'[./]', '', regex=True)
    
    # 同票人数 (同行人数)
    ticket_counts = df['Ticket'].value_counts()
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)
    
    # === 5. Cabin 特征 ===
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['CabinDeck'] = df['Cabin'].str[0].fillna('U')  # U = Unknown
    df['CabinNum'] = df['Cabin'].str.extract(r'(\d+)').astype(float)
    
    # Cabin 中的房间数
    df['CabinCount'] = df['Cabin'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
    
    # === 6. 年龄特征 ===
    # 用更多维度填充年龄
    age_fill = df.groupby(['Title', 'Pclass', 'Sex'])['Age'].transform('median')
    df['Age'] = df['Age'].fillna(age_fill)
    age_fill2 = df.groupby(['Title', 'Pclass'])['Age'].transform('median')
    df['Age'] = df['Age'].fillna(age_fill2)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    
    # 年龄相关特征
    df['IsChild'] = (df['Age'] < 12).astype(int)
    df['IsElderly'] = (df['Age'] > 60).astype(int)
    df['Age*Class'] = df['Age'] * df['Pclass']
    
    # === 7. 票价特征 ===
    df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # 人均票价 (同行分摊)
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']
    
    # 票价对数
    df['FareLog'] = np.log1p(df['Fare'])
    
    # 票价分箱
    df['FareBin'] = pd.qcut(df['Fare'], 5, labels=False, duplicates='drop')
    
    # === 8. 登船港口 ===
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # === 9. 交互特征 ===
    df['Sex*Pclass'] = df['Sex'].map({'male': 0, 'female': 1}).astype(str) + '_' + df['Pclass'].astype(str)
    df['Title*Pclass'] = df['Title'] + '_' + df['Pclass'].astype(str)
    
    # === 10. 编码 ===
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    
    # One-hot 编码
    cat_cols = ['Embarked', 'Title', 'FamilyCat', 'CabinDeck', 'Sex*Pclass', 'Title*Pclass']
    for col in cat_cols:
        dummies = pd.get_dummies(df[col], prefix=col, dtype=int)
        df = pd.concat([df, dummies], axis=1)
    
    return df

full_fe = advanced_feature_engineering(full)

In [ ]:
# 选择特征
feature_cols = [
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'FamilySize', 'IsAlone', 'HasCabin', 'CabinNum', 'CabinCount',
    'IsChild', 'IsElderly', 'Age*Class', 'FarePerPerson', 'FareLog', 'FareBin',
    'TicketGroupSize', 'NameLen'
]

# 添加 one-hot 列
embarked_cols = [c for c in full_fe.columns if c.startswith('Embarked_')]
title_cols = [c for c in full_fe.columns if c.startswith('Title_')]
family_cols = [c for c in full_fe.columns if c.startswith('FamilyCat_')]
deck_cols = [c for c in full_fe.columns if c.startswith('CabinDeck_')]
sp_cols = [c for c in full_fe.columns if c.startswith('Sex*Pclass_')]
tp_cols = [c for c in full_fe.columns if c.startswith('Title*Pclass_')]

feature_cols += embarked_cols + title_cols + family_cols + deck_cols + sp_cols + tp_cols

# 分割
train_fe = full_fe[full_fe['is_train'] == 1].copy()
test_fe = full_fe[full_fe['is_train'] == 0].copy()

X_train = train_fe[feature_cols].fillna(0)
y_train = train_fe['Survived'].astype(int)
X_test = test_fe[feature_cols].fillna(0)

print(f'Features: {len(feature_cols)}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'NaN in train: {X_train.isnull().sum().sum()}')
print(f'NaN in test: {X_test.isnull().sum().sum()}')

## 2. 超参调优

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Random Forest 调参
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3]
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
rf_grid.fit(X_train, y_train)
print(f'RF Best: {rf_grid.best_score_:.4f} | {rf_grid.best_params_}')

# Gradient Boosting 调参
gb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_leaf': [1, 3, 5]
}
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train, y_train)
print(f'GB Best: {gb_grid.best_score_:.4f} | {gb_grid.best_params_}')

# SVM (需要标准化)
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42))
])
svm_params = {
    'svm__C': [0.5, 1, 5],
    'svm__kernel': ['rbf', 'poly'],
    'svm__gamma': ['scale', 'auto']
}
svm_grid = GridSearchCV(svm_pipe, svm_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
svm_grid.fit(X_train, y_train)
print(f'SVM Best: {svm_grid.best_score_:.4f} | {svm_grid.best_params_}')

In [ ]:
# KNN
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
knn_params = {'knn__n_neighbors': [3, 5, 7, 9], 'knn__weights': ['uniform', 'distance']}
knn_grid = GridSearchCV(knn_pipe, knn_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
knn_grid.fit(X_train, y_train)
print(f'KNN Best: {knn_grid.best_score_:.4f} | {knn_grid.best_params_}')

# Logistic Regression
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
lr_params = {'lr__C': [0.1, 0.5, 1, 5], 'lr__penalty': ['l1', 'l2'], 'lr__solver': ['liblinear']}
lr_grid = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
lr_grid.fit(X_train, y_train)
print(f'LR Best: {lr_grid.best_score_:.4f} | {lr_grid.best_params_}')

## 3. 模型融合

In [ ]:
# 收集最优模型
best_rf = rf_grid.best_estimator_
best_gb = gb_grid.best_estimator_
best_svm = svm_grid.best_estimator_
best_knn = knn_grid.best_estimator_
best_lr = lr_grid.best_estimator_

# --- Voting Ensemble ---
voting = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('gb', best_gb),
        ('svm', best_svm),
        ('lr', best_lr),
        ('knn', best_knn)
    ],
    voting='soft'
)
voting_scores = cross_val_score(voting, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Voting Ensemble CV: {voting_scores.mean():.4f} (+/- {voting_scores.std():.4f})')

# --- Stacking ---
stacking = StackingClassifier(
    estimators=[
        ('rf', best_rf),
        ('gb', best_gb),
        ('svm', best_svm),
        ('lr', best_lr)
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5
)
stacking_scores = cross_val_score(stacking, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Stacking Ensemble CV: {stacking_scores.mean():.4f} (+/- {stacking_scores.std():.4f})')

## 4. 生成最终提交

In [ ]:
# 对比所有模型
print('\n=== Model Comparison ===')
print(f'RF CV:      {rf_grid.best_score_:.4f}')
print(f'GB CV:      {gb_grid.best_score_:.4f}')
print(f'SVM CV:     {svm_grid.best_score_:.4f}')
print(f'KNN CV:     {knn_grid.best_score_:.4f}')
print(f'LR CV:      {lr_grid.best_score_:.4f}')
print(f'Voting CV:  {voting_scores.mean():.4f}')
print(f'Stacking CV:{stacking_scores.mean():.4f}')

In [ ]:
# 用最佳融合模型生成提交
best_model = stacking if stacking_scores.mean() > voting_scores.mean() else voting
best_name = 'Stacking' if stacking_scores.mean() > voting_scores.mean() else 'Voting'

best_model.fit(X_train, y_train)
predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions.astype(int)
})
submission.to_csv('../submission_v2.csv', index=False)
print(f'Best model: {best_name}')
print(f'Submission saved: {submission.shape}')
submission.head()